In [1]:
import os

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from torch.nn.functional import softmax
from torch.optim import AdamW
from torch.utils.data import DataLoader

from internal.data_types import HistologyDataset
from internal.nn.model import train_one_epoch, validate
from internal.persistence_manager import PersistenceManager
from notebooks.internal.nn.weighted_random_sampler import make_weighted_sampler

data = PersistenceManager.load_dataset()
test_df = data.test_df
train_df = data.train_df
train_transforms = data.train_transforms
val_test_transforms = data.val_test_transforms
idx2label = data.idx2label

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_is_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [2]:
N_FOLDS = data.num_K_folds
IMAGE_SIZE = data.image_size
BATCH_SIZE = 4
PRETRAINED_MODEL = "tf_efficientnetv2_s.in21k"
N_CLASSES = 4 # number of classes in the dataset (labels)
N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4

for fold in range(N_FOLDS):
    print(f"\n========== Fold {fold} ==========")

    train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
    val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

    train_dataset = HistologyDataset(train_df_split, transforms=train_transforms, is_train=True, image_size=IMAGE_SIZE)
    val_dataset   = HistologyDataset(val_df_split,   transforms=val_test_transforms, is_train=True, image_size=IMAGE_SIZE)

    sampler = make_weighted_sampler(train_df_split)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=N_WORKERS,
        pin_memory=cuda_is_available
    )
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=N_WORKERS, pin_memory=cuda_is_available)

    # --- create fresh model for this fold ---
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=True,
        num_classes=N_CLASSES
    ).to(device)

    # --- Stage 1: freeze backbone, train classifier head ---
    print("\n--- Stage 1: Training classifier head ---")

    # --- 1.1. freeze feature extractor layers ---
    for param in model.parameters():
        param.requires_grad = False

    # 2) unfreeze classifier head (EffNetV2 uses .classifier)
    for param in model.classifier.parameters():
        param.requires_grad = True

    # --- 1.2. define loss, optimizer, scheduler ---
    criterion = nn.CrossEntropyLoss()
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=10
    )

    # --- 1.3. train for several epochs ---
    EPOCHS = 8
    best_f1 = 0.0
    best_state = None
    for epoch in range(1, EPOCHS+1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc, val_f1 = validate(
            model, val_loader, criterion, device
        )
        scheduler.step()
        print(
            f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
            f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
        )
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict().copy()
            torch.save(best_state, f"best_effv2_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
            print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

    # --- Stage 2: unfreeze whole model, fine-tune ---
    print("\n--- Stage 2: Fine-tuning entire model ---")

    # --- 2.1. unfreeze entire model ---
    for param in model.parameters():
        param.requires_grad = True

    # --- 2.2. define loss, optimizer, scheduler ---
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=15
    )

    # --- 2.3. mild class weights ---
    class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
    class_weights = (class_counts.sum() / class_counts)
    class_weights = class_weights / class_weights.mean()
    # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

    # --- 2.4. train for several epochs ---
    EPOCHS = 15
    best_f1 = 0.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc, val_f1 = validate(
            model, val_loader, criterion, device
        )
        scheduler.step()
        print(
            f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
            f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
        )
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict().copy()
            torch.save(best_state, f"best_effv2_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
            print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

    if best_state is not None:
        model.load_state_dict(best_state)   # restore best val-F1 weights

    # --- save model for this fold ---
    torch.save(model.state_dict(), f"effv2_s_fold{fold}.pth")


========== Fold 0 ==========

--- Stage 1: Training classifier head ---

Epoch 1/8


    t_loss=3.9448 | F1(macro)=0.2748 | Acc=0.2781


Confusion matrix:
 [[31 43  4 11]
 [18 34 18 13]
 [27 30 13  9]
 [ 7 18  3  4]]
Train  loss=3.9448 acc=0.2781 f1=0.2748 | Val loss=5.5459 acc=0.2898 f1=0.2564
  🔥 New best F1: 0.2564 – model saved.

Epoch 2/8


    t_loss=3.4874 | F1(macro)=0.3113 | Acc=0.3109


Confusion matrix:
 [[30 34  9 16]
 [27 31 13 12]
 [22 25 14 18]
 [ 4 20  4  4]]
Train  loss=3.4874 acc=0.3109 f1=0.3113 | Val loss=5.3407 acc=0.2792 f1=0.2507

Epoch 3/8


...

Confusion matrix:
 [[27 13  9 40]
 [23 23  9 28]
 [22 12 15 30]
 [10  9  1 11]]
Train  loss=1.0346 acc=0.5673 f1=0.5611 | Val loss=1.7799 acc=0.2695 f1=0.2667

Epoch 14/15


    t_loss=1.0157 | F1(macro)=0.5428 | Acc=0.5531


Confusion matrix:
 [[32  6 10 41]
 [29 16 13 25]
 [24  2 26 27]
 [12  3  4 12]]
Train  loss=1.0157 acc=0.5531 f1=0.5428 | Val loss=1.7075 acc=0.3050 f1=0.3014

Epoch 15/15


    t_loss=0.9912 | F1(macro)=0.5753 | Acc=0.5850


Confusion matrix:
 [[25 15 10 39]
 [27 22  9 25]
 [23  9 22 25]
 [10  5  4 12]]
Train  loss=0.9912 acc=0.5850 f1=0.5753 | Val loss=1.7171 acc=0.2872 f1=0.2881


In [3]:
test_dataset = HistologyDataset(
    test_df,
    transforms=val_test_transforms,
    is_train=False,   # returns (img, sample_index)
    image_size=IMAGE_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=cuda_is_available
)

all_fold_probs = []   # list of arrays [N, num_classes]
all_sample_indices = None

for fold in range(N_FOLDS):
    print(f"Inference with fold {fold} model")

    # recreate model and load weights
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=False,
        num_classes=N_CLASSES
    ).to(device)
    state = torch.load(f"effv2_s_fold{fold}.pth", map_location=device)
    model.load_state_dict(state)
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for imgs, sample_indices in test_loader:
            imgs = imgs.to(device, non_blocking=True)

            logits = model(imgs)               # [B, num_classes]
            probs = softmax(logits, dim=1)     # [B, num_classes]
            fold_probs.append(probs.cpu().numpy())

            # collect sample indices only once
            if all_sample_indices is None:
                sample_indices_list.extend(sample_indices)

    fold_probs = np.concatenate(fold_probs, axis=0)  # [N, num_classes]
    all_fold_probs.append(fold_probs)

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

# average probabilities across folds
mean_probs = np.mean(all_fold_probs, axis=0)   # [N, num_classes]
pred_indices = mean_probs.argmax(axis=1)

pred_labels = [idx2label[int(i)] for i in pred_indices]
sample_index_with_ext = [
    f"{si}.png" if not si.endswith(".png") else si
    for si in all_sample_indices
]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})

submission_df.to_csv("submission_5fold_no_tta.csv", index=False)
print("Saved submission_5fold_no_tta.csv")
print(submission_df.head())


Inference with fold 0 model
Inference with fold 1 model
Inference with fold 2 model
Inference with fold 3 model
Inference with fold 4 model
Saved submission_5fold_no_tta.csv
   sample_index            label
0  img_0000.png          HER2(+)
1  img_0001.png  Triple negative
2  img_0002.png  Triple negative
3  img_0003.png  Triple negative
4  img_0004.png        Luminal B


In [4]:
########################################################
# ===== Inference with TTA and 5-Fold Ensembling ===== #
########################################################
# all_fold_probs = []
# all_sample_indices = None
#
# test_dataset = HistologyDataset(test_df, transforms=val_test_transforms, is_train=False, image_size=IMAGE_SIZE)
# test_loader = DataLoader(test_dataset, batch_size=1,  # IMPORTANT: batch_size=1 for per-image TTA
#                          shuffle=False, num_workers=N_WORKERS, pin_memory=cuda_is_available)
#
# for fold in range(N_FOLDS):
#     print(f"Inference with fold {fold} model")
#     model = timm.create_model(PRETRAINED_MODEL, pretrained=False, num_classes=N_CLASSES).to(device)
#     model.load_state_dict(torch.load(f"effv2_s_fold{fold}.pth", map_location=device))
#     model.eval()
#
#     fold_probs = []
#     sample_indices_list = []
#
#     with torch.no_grad():
#         for img_tensor, sample_idx in test_loader:
#             img_tensor = img_tensor.squeeze(0)  # [3,H,W]
#             img_tensor = img_tensor.to(device)
#
#             # -------- TTA: apply multiple augmented views --------
#             tta_tensors = apply_tta(img_tensor)
#
#             # accumulate probability predictions
#             probs_sum = 0
#             for aug_img in tta_tensors:
#                 aug_img = aug_img.unsqueeze(0).to(device)  # [1,3,H,W]
#                 logits = model(aug_img)
#                 probs = softmax(logits, dim=1)  # [1,4]
#                 probs_sum += probs[0].cpu().numpy()
#
#             # average across TTA views
#             avg_probs = probs_sum / len(tta_tensors)
#             fold_probs.append(avg_probs)
#
#             if all_sample_indices is None:
#                 sample_indices_list.append(sample_idx[0])
#
#     fold_probs = np.vstack(fold_probs)  # [N, 4]
#     all_fold_probs.append(fold_probs)
#
#     if all_sample_indices is None:
#         all_sample_indices = sample_indices_list
#
# mean_probs = np.mean(all_fold_probs, axis=0)  # [N, 4]
# pred_indices = mean_probs.argmax(axis=1)
# pred_labels = [idx2label[int(i)] for i in pred_indices]
#
# sample_index_with_ext = [
#     f"{si}.png" if not si.endswith(".png") else si
#     for si in all_sample_indices
# ]
#
# submission_df = pd.DataFrame({
#     "sample_index": sample_index_with_ext,
#     "label": pred_labels
# })
# submission_df.to_csv("submission_5fold_tta.csv", index=False)
#
# print("Saved submission_5fold_tta.csv")

In [5]:
from internal.nn.test_time_augmentation import apply_tta
from sklearn.metrics import f1_score
from torch.nn.functional import softmax

def predict_loader_with_tta(model, loader, device):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in loader:  # note: here we have labels, not sample_index
            imgs = imgs.squeeze(0).to(device)  # if batch_size=1
            tta_imgs = apply_tta(imgs)         # same apply_tta as for test

            probs_sum = 0
            for aug in tta_imgs:
                aug = aug.unsqueeze(0).to(device)
                logits = model(aug)
                probs = softmax(logits, dim=1)
                probs_sum += probs[0].cpu().numpy()

            avg_probs = probs_sum / len(tta_imgs)
            all_probs.append(avg_probs)
            all_targets.append(labels.item())

    all_probs = np.vstack(all_probs)
    all_targets = np.array(all_targets)
    pred_indices = all_probs.argmax(axis=1)

    macro_f1 = f1_score(all_targets, pred_indices, average="macro")
    return macro_f1

fold_f1s = []

for fold in range(N_FOLDS):
    print(f"OOF eval for fold {fold}")

    # build val_df_split for that fold
    val_df_split = train_df[train_df["fold"] == fold].reset_index(drop=True)
    val_dataset = HistologyDataset(val_df_split, transforms=val_test_transforms, is_train=True, image_size=IMAGE_SIZE)
    val_loader  = DataLoader(val_dataset, batch_size=1, shuffle=False,
                             num_workers=N_WORKERS, pin_memory=cuda_is_available)

    model = timm.create_model(PRETRAINED_MODEL, pretrained=False, num_classes=N_CLASSES).to(device)
    model.load_state_dict(torch.load(f"effv2_s_fold{fold}.pth", map_location=device))

    f1 = predict_loader_with_tta(model, val_loader, device)
    fold_f1s.append(f1)
    print("Fold F1 (OOF, with TTA):", f1)

print("Mean OOF F1:", np.mean(fold_f1s))


OOF eval for fold 0
Fold F1 (OOF, with TTA): 0.23317986778352634
OOF eval for fold 1
Fold F1 (OOF, with TTA): 0.28524557719067734
OOF eval for fold 2
Fold F1 (OOF, with TTA): 0.3191490693499622
OOF eval for fold 3
Fold F1 (OOF, with TTA): 0.2706382331996765
OOF eval for fold 4
Fold F1 (OOF, with TTA): 0.29324884792626726
Mean OOF F1: 0.2802923190900219
